# MVP — qual assunto o agente conversa, por segmento

Motor de decisão para o fluxo de ativação de conta: para cada cliente novo que entra
no fluxo do agente, escolher **sobre qual dos quatro assuntos conversar** (pix,
pagamento, seguro, investimento), aprendendo com a ativação em D7.

## A decisão de desenho que define este notebook

O CATE de um assunto é

$$\text{CATE}(X, a) = \underbrace{\mathbb{E}[Y \mid X, a]}_{\text{depende do assunto}} - \underbrace{\mathbb{E}[Y \mid X, \text{silêncio}]}_{\text{NÃO depende do assunto}}$$

O segundo termo é constante em relação ao braço. Logo

$$\arg\max_a \text{CATE}(X, a) \;=\; \arg\max_a \mathbb{E}[Y \mid X, a]$$

**Escolher entre assuntos não precisa de contrafactual nenhum** — precisa de alocação
randomizada e da conversão observada por braço. Toda a maquinaria causal (transformed
outcome $Y^*$, IPW, propensão por braço, braço de controle interno) serve para
responder *uma* pergunta a mais: *o melhor assunto vence o silêncio?*

Aqui essa pergunta já tem dono: **o holdout aleatório de 20%**. O agente conversa com
todo mundo que entra no fluxo — o que se controla é o assunto, não se fala ou não.
Manter um controle interno *além* do holdout privaria ~32% dos clientes de tratamento,
com dois grupos de controle que medem coisas diferentes.

Então este MVP é:

| camada | mecanismo | responde |
|---|---|---|
| holdout 20% aleatório | diferença de proporções, global e por segmento | o programa funciona? estamos machucando algum segmento? |
| fluxo 80% | Thompson Sampling Beta-Bernoulli por `(segmento, assunto)` | qual assunto para quem? |

Sem XGBoost, sem IPW, sem $Y^*$. A política inteira é **uma tabela** de
`n_segmentos × n_assuntos` probabilidades — versionável, auditável e servível de um
arquivo de config, sem modelo no caminho da requisição.

O caminho de volta para a versão causal está na última seção.

In [ ]:
import json

import numpy as np
import pandas as pd

ASSUNTOS = ["pix", "pagamento", "seguro", "investimento"]
HOLDOUT = "holdout"

# Números de contexto.md — usados como prior e para dimensionamento.
TAXA_D7 = 0.18          # ativação em D7 sem intervenção
CLIENTES_SEMANA = 29_000
FRACAO_HOLDOUT = 0.20

## 1. O contrato de segmentação

O segmento é a única "feature" do modelo. Isso é deliberado: com 4 assuntos e ~23k
clientes por semana no fluxo, a granularidade que os dados sustentam é dezenas de
células, não um modelo em features contínuas (a seção 6 quantifica).

Duas regras que a API abaixo força:

- **O segmento é calculado no momento da decisão e gravado no log.** Recalcular no job
  de treino a partir do cadastro de hoje vaza futuro: renda e engajamento mudaram *por
  causa* da conversa. O `update()` exige a coluna `segmento` e nunca reprocessa features.
- **Lista fechada.** Um segmento novo aparecendo no scoring é erro, não uma linha a
  mais na tabela — a política é um artefato versionado, não um dicionário que cresce.

Aqui as faixas são de exemplo. Na prática, defina os cortes com o **histórico
pré-agente**, que é exatamente para isso que ele serve: ele diz qual perfil ativa por
qual rota naturalmente, e é bom para desenhar segmento. Ele **não** diz qual assunto
conversar — propensão natural não é resposta a mensagem. Priors iguais para todos os
braços, e deixe o bandit descobrir.

In [ ]:
SEGMENTOS = [
    "jovem_renda_baixa",  "jovem_renda_alta",
    "adulto_renda_baixa", "adulto_renda_alta",
    "senior_renda_baixa", "senior_renda_alta",
]


def segmentar(df):
    """Contexto -> segmento. Chamado NA DECISÃO; o resultado vai para o log."""
    idade = pd.cut(df["idade"], bins=[0, 29, 49, 200],
                   labels=["jovem", "adulto", "senior"]).astype(str)
    renda = np.where(df["renda"] > 5000, "renda_alta", "renda_baixa")
    return pd.Series(idade + "_" + renda, index=df.index, name="segmento")

## 2. O motor

Thompson Sampling com posterior Beta-Bernoulli em cada célula `(segmento, assunto)`.
São duas contagens por célula: sucessos e falhas. É isso.

Quatro pontos de desenho que valem o comentário:

- **A política é uma tabela, não um modelo.** Como os segmentos são discretos, a
  distribuição de servimento é a mesma para todo cliente do segmento. Então
  `tabela_politica()` calcula $P(\text{braço é o melhor})$ por Monte Carlo uma vez por
  atualização — não uma vez por requisição. O que vai para produção é um CSV de
  6 × 4 números. Isso é o que torna o MVP auditável: dá para colocar a tabela num slide.

- **Propensão exata no log.** Como a distribuição vem da tabela, a probabilidade com
  que cada ação foi sorteada é *conhecida*, não estimada. Logamos a linha inteira
  (todos os braços), não só a do braço sorteado. É barato e é o que permite análise
  off-policy depois e migrar para a versão causal sem refazer histórico.

- **Prior fracamente informativo em 18%.** `Beta(1,1)` é uniforme em [0,1] e faz o TS
  se debater nas primeiras semanas testando a hipótese de que um assunto converte 80%.
  Ancorar em `TAXA_D7` com força de ~20 observações mata esse ruído e é irrelevante
  depois da primeira semana (~900 obs/célula/semana). Como o prior é **igual em todos
  os braços**, ele não enviesa a escolha.

- **Piso por braço.** O TS naturalmente reduz um braço ruim a ~0,1% do tráfego, o que
  na prática o mata: se ele melhorar (produto novo, sazonalidade), ninguém descobre. O
  piso garante medição contínua. Ele custa conversão — a seção 6 mostra quanto.

In [ ]:
class BanditSegmentado:
    """
    Thompson Sampling Beta-Bernoulli por (segmento, assunto).

    Escolhe SOBRE O QUE conversar. Não decide se conversa — isso é o holdout.
    Estado = duas matrizes de contagem (n_segmentos, n_assuntos). A política
    servida é a tabela devolvida por tabela_politica().
    """

    def __init__(self, assuntos, segmentos, funcao_segmento,
                 taxa_prior=TAXA_D7, forca_prior=20.0, piso_por_braco=0.05,
                 decay=1.0, n_mc=4000, random_state=None):
        """
        taxa_prior/forca_prior : prior Beta com essa média e esse peso em
                                 observações. Igual para todos os braços.
        piso_por_braco         : probabilidade mínima de servimento de cada
                                 assunto em cada segmento. Garante que nenhum
                                 braço morra e que a medição continue.
        decay                  : fator aplicado às contagens antes de somar as
                                 novas (esquecimento exponencial). 1.0 = nunca
                                 esquece. Use ~0.97/semana se o efeito for
                                 sazonal ou se a mensagem em si mudar.
        n_mc                   : amostras de Monte Carlo para P(braço é o melhor).
                                 Roda uma vez por atualização, não por cliente.
        """
        if not assuntos or not segmentos:
            raise ValueError("assuntos e segmentos não podem ser vazios")
        eps = piso_por_braco * len(assuntos)
        if not 0 <= eps <= 1:
            raise ValueError(
                f"piso_por_braco * n_assuntos = {eps:.2f}; tem que caber em [0, 1]"
            )

        self.assuntos = list(assuntos)
        self.segmentos = list(segmentos)
        self.funcao_segmento = funcao_segmento
        self.piso_por_braco = piso_por_braco
        self.decay = decay
        self.n_mc = n_mc
        self.rng = np.random.default_rng(random_state)

        self.alpha0 = taxa_prior * forca_prior
        self.beta0 = (1.0 - taxa_prior) * forca_prior

        forma = (len(self.segmentos), len(self.assuntos))
        self.succ = np.zeros(forma)   # conversões por célula
        self.fail = np.zeros(forma)   # não-conversões por célula
        self._i_seg = {s: i for i, s in enumerate(self.segmentos)}
        self._i_arm = {a: j for j, a in enumerate(self.assuntos)}

    @property
    def prop_cols(self):
        return [f"p_{a}" for a in self.assuntos]

    # ------------------------------------------------------------------ #
    # Posterior e política
    # ------------------------------------------------------------------ #
    def posterior(self):
        """(alpha, beta) de cada célula. Beta(a, b) sobre a taxa de ativação D7."""
        return self.alpha0 + self.succ, self.beta0 + self.fail

    def prob_melhor(self):
        """
        P(assunto é o melhor do segmento | dados), por Monte Carlo.

        É exatamente a probabilidade com que o Thompson Sampling escolheria cada
        braço. Calcular a distribuição inteira (em vez de sortear um theta por
        cliente) é o que permite logar a propensão exata e servir de tabela.
        """
        a, b = self.posterior()
        draws = self.rng.beta(a[None, :, :], b[None, :, :], size=(self.n_mc, *a.shape))
        vencedor = draws.argmax(axis=2)                       # (n_mc, n_segmentos)
        contagem = np.stack([
            np.bincount(vencedor[:, s], minlength=len(self.assuntos))
            for s in range(len(self.segmentos))
        ])
        return contagem / self.n_mc

    def tabela_politica(self):
        """
        A POLÍTICA: probabilidade de servir cada assunto em cada segmento.

            p = (1 - eps) * P(é o melhor) + eps / n_assuntos,  eps = piso * n_assuntos

        A mistura garante p >= piso_por_braco para todo braço, exatamente.
        Este DataFrame é o artefato de deploy.
        """
        eps = self.piso_por_braco * len(self.assuntos)
        probs = (1 - eps) * self.prob_melhor() + eps / len(self.assuntos)
        return pd.DataFrame(probs, columns=self.assuntos,
                            index=pd.Index(self.segmentos, name="segmento"))

    def melhor_assunto(self):
        """Assunto de maior média posterior por segmento. Explotação pura, p/ auditoria."""
        a, b = self.posterior()
        return pd.Series(np.array(self.assuntos)[(a / (a + b)).argmax(axis=1)],
                         index=pd.Index(self.segmentos, name="segmento"),
                         name="melhor_assunto")

    # ------------------------------------------------------------------ #
    # Decisão
    # ------------------------------------------------------------------ #
    def recommend_batch(self, contextos, tabela=None):
        """
        Sorteia um assunto por cliente.

        Devolve (ações, decisão), onde `decisão` tem o segmento e a propensão de
        TODOS os braços — é o que precisa ir para o log junto com a ação.
        Passe `tabela` para garantir que o lote inteiro decidiu com a mesma
        política (em produção: a tabela da versão que está no ar).
        """
        seg = self.funcao_segmento(contextos)
        desconhecidos = set(seg.unique()) - set(self._i_seg)
        if desconhecidos:
            raise ValueError(f"segmentos fora do contrato: {sorted(desconhecidos)}")

        tabela = self.tabela_politica() if tabela is None else tabela
        p = tabela.loc[seg.to_numpy(), self.assuntos].to_numpy()
        p = p / p.sum(axis=1, keepdims=True)      # blinda contra ponto flutuante

        u = self.rng.random((p.shape[0], 1))
        escolhido = (u > p.cumsum(axis=1)).sum(axis=1)
        acoes = np.array(self.assuntos)[escolhido]

        decisao = pd.DataFrame(p, columns=self.prop_cols, index=seg.index)
        decisao.insert(0, "segmento", seg.to_numpy())
        return acoes, decisao

    def recommend(self, contexto):
        """Caminho de produção, um cliente. contexto: dict ou DataFrame de 1 linha."""
        df = pd.DataFrame([contexto]) if isinstance(contexto, dict) else contexto
        acoes, decisao = self.recommend_batch(df)
        return acoes[0], decisao.iloc[0].to_dict()

    # ------------------------------------------------------------------ #
    # Treino
    # ------------------------------------------------------------------ #
    def update(self, log, verbose=True):
        """
        Soma as contagens de uma coorte JÁ MADURA (7 dias completos).

        Exige as colunas ['segmento', 'acao', 'converteu']. O segmento vem do log
        — o que foi gravado na decisão — e nunca é recalculado aqui: features de
        hoje já sofreram efeito da conversa. Linhas de holdout (ou de qualquer
        ação fora do contrato) são ignoradas e contadas no relatório.
        """
        faltando = [c for c in ["segmento", "acao", "converteu"] if c not in log.columns]
        if faltando:
            raise ValueError(f"colunas ausentes no log: {faltando}")

        self.succ *= self.decay
        self.fail *= self.decay

        g = log.groupby(["segmento", "acao"], observed=True)["converteu"].agg(["sum", "count"])
        ignoradas = 0
        for (seg, arm), r in g.iterrows():
            if seg not in self._i_seg or arm not in self._i_arm:
                ignoradas += int(r["count"])
                continue
            i, j = self._i_seg[seg], self._i_arm[arm]
            self.succ[i, j] += float(r["sum"])
            self.fail[i, j] += float(r["count"] - r["sum"])

        if verbose:
            n_tot = int((self.succ + self.fail).sum())
            print(f"update: +{len(log)} linhas ({ignoradas} fora do fluxo) "
                  f"| base acumulada: {n_tot}")
        return self

    # ------------------------------------------------------------------ #
    # Auditoria
    # ------------------------------------------------------------------ #
    def estado(self):
        """Uma linha por célula: contagens, posterior, P(é o melhor) e p de servimento."""
        a, b = self.posterior()
        pm, tab = self.prob_melhor(), self.tabela_politica().to_numpy()
        n = self.succ + self.fail
        soma = a + b
        return pd.DataFrame([
            dict(segmento=s, assunto=arm,
                 n=n[i, j], conv=self.succ[i, j],
                 taxa=self.succ[i, j] / n[i, j] if n[i, j] else np.nan,
                 post_media=a[i, j] / soma[i, j],
                 post_sd=np.sqrt(a[i, j] * b[i, j] / (soma[i, j] ** 2 * (soma[i, j] + 1))),
                 p_melhor=pm[i, j], p_serve=tab[i, j])
            for i, s in enumerate(self.segmentos)
            for j, arm in enumerate(self.assuntos)
        ])

## 3. Simulador com a verdade conhecida

Rodar sem erro não prova nada — com `converteu` aleatório o notebook imprime tabelas
igualmente bonitas. Para afirmar que o motor funciona é preciso um mundo com o efeito
verdadeiro conhecido.

**Os efeitos aqui são de magnitude realista: 0,3 a 2,5 p.p.** Isso importa mais do que
parece. É comum validar bandit num simulador com uplift de 10-15 p.p., onde qualquer
estimador acerta; sob efeito de 1-2 p.p. em cima de uma base de 18%, muita coisa que
"funcionava" para de funcionar. Os números abaixo assumem o cenário difícil.

A verdade tem heterogeneidade real (o melhor assunto muda por segmento), um assunto que
é fraco em quase todo lugar (`seguro`) e casos em que um assunto **atrapalha** — uplift
negativo. Note também que a **taxa base varia por segmento**: isso é inofensivo aqui,
porque a comparação entre assuntos é sempre *dentro* do segmento, onde a base é comum.
É a mesma razão pela qual o termo do controle cancela no argmax.

In [ ]:
# Uplift VERDADEIRO em pontos percentuais, vs. não conversar sobre nada.
# É isso que o bandit tem que redescobrir sozinho.
UPLIFT_PP = pd.DataFrame(
    [[+2.5, +1.0, -0.5, -0.5],
     [+2.0, +0.8, -0.3, +1.2],
     [+0.8, +2.2, +0.3, -0.4],
     [+0.9, +1.4, +0.6, +2.0],
     [+0.3, +1.6, +1.0, -0.3],
     [+0.4, +1.0, +1.2, +2.4]],
    index=pd.Index(SEGMENTOS, name="segmento"), columns=ASSUNTOS,
)

# Taxa de ativação D7 SEM intervenção, por segmento (média ponderada ~= 18%).
BASE = pd.Series([0.14, 0.19, 0.13, 0.20, 0.12, 0.19],
                 index=pd.Index(SEGMENTOS, name="segmento"))


class Simulador:
    """Mundo sintético com CATE verdadeiro conhecido e magnitude realista."""

    def __init__(self, seed=42):
        self.rng = np.random.default_rng(seed)
        self.uplift = UPLIFT_PP / 100.0

    def gerar(self, n):
        df = pd.DataFrame({
            "idade": self.rng.integers(18, 65, n),
            "renda": self.rng.uniform(1000, 15000, n),
        })
        df["segmento"] = segmentar(df)
        return df

    def _taxa(self, seg, acoes):
        seg, acoes = np.asarray(seg), np.asarray(acoes)
        efeito = np.zeros(len(seg))
        for a in ASSUNTOS:                       # HOLDOUT fica com efeito 0
            m = acoes == a
            if m.any():
                efeito[m] = self.uplift[a].reindex(seg[m]).to_numpy()
        return BASE.reindex(seg).to_numpy() + efeito

    def taxa_esperada(self, seg, acoes):
        """E[ativação] de uma política, sem ruído de amostragem."""
        return float(self._taxa(seg, acoes).mean())

    def simular(self, seg, acoes):
        """Joga a moeda: ativou em D7 ou não."""
        p = np.clip(self._taxa(seg, acoes), 1e-4, 1 - 1e-4)
        return (self.rng.random(len(p)) < p).astype(int)

## 4. Validação 1 — quanto a segmentação compra?

Antes de rodar bandit nenhum: se o melhor assunto fosse o mesmo para todo mundo, um
teste A/B de 4 braços resolveria e este notebook não teria razão de existir. Então a
primeira pergunta é o tamanho do prêmio pela personalização.

Os três patamares: o **oráculo segmentado** (teto), o **melhor assunto único** (o que um
A/B tradicional entregaria) e o **pior assunto** (o custo de escolher errado).

In [ ]:
sim_pop = Simulador(1)
pop = sim_pop.gerar(400_000)
peso = pop["segmento"].value_counts(normalize=True).reindex(SEGMENTOS)

ate_global = (UPLIFT_PP.T * peso).T.sum()          # ATE de cada assunto na população
teto = float((UPLIFT_PP.max(axis=1) * peso).sum())  # oráculo por segmento
melhor_unico = ate_global.idxmax()

print("Distribuição da população e melhor assunto verdadeiro por segmento")
print(pd.DataFrame({
    "peso": peso.round(4),
    "base_D7": BASE,
    "melhor": UPLIFT_PP.idxmax(axis=1),
    "uplift_pp": UPLIFT_PP.max(axis=1),
    "pior_pp": UPLIFT_PP.min(axis=1),
}).to_string())

print("\nATE de cada assunto na população inteira (p.p.)")
print(ate_global.round(3).to_string())

print(f"\n{'política':<34}{'uplift':>10}{'ativações/semana':>19}")
por_semana = lambda pp: pp / 100 * CLIENTES_SEMANA * (1 - FRACAO_HOLDOUT)
for nome, pp in [
    ("Oráculo segmentado (teto)", teto),
    (f"Melhor assunto único ('{melhor_unico}')", ate_global.max()),
    ("Assunto aleatório", float(ate_global.mean())),
    (f"Pior assunto único ('{ate_global.idxmin()}')", ate_global.min()),
]:
    print(f"{nome:<34}{pp:>+9.2f}pp{por_semana(pp):>19,.0f}")

print(f"\nO prêmio da personalização: +{teto - ate_global.max():.2f} pp "
      f"({teto / ate_global.max() - 1:.0%} a mais que o melhor assunto único)")
print(f"= +{por_semana(teto - ate_global.max()):,.0f} ativações/semana. No teto, o "
      f"programa move o OKR de {TAXA_D7:.1%} para {TAXA_D7 + teto / 100:.1%} "
      f"({teto / 100 / TAXA_D7:.1%} de lift relativo) na população tratada.")

## 5. Validação 2 — o ciclo semanal, com a maturação de 7 dias

Cada rodada é uma semana de aberturas de conta. A restrição operacional que molda tudo:
**o resultado de uma coorte só existe 7 dias depois**. Então a coorte da semana 1 só
entra no treino no fim da semana 2, e as semanas 1 e 2 decidem no prior. Não há como
contornar isso; o que dá é não fingir que não existe.

Duas métricas, porque medem coisas diferentes:

| coluna | o que é |
|---|---|
| `realizado` | conversão da política que **rodou de fato** — já paga o piso de exploração. É o que o negócio vê essa semana. |
| `aprendido` | conversão se você congelasse o melhor assunto de cada segmento agora. É o que o modelo **sabe**. |

O intervalo entre as duas é o preço da exploração, e não desaparece: é ele que mantém a
medição viva. A seção 6 mostra o tamanho exato desse pedágio.

In [ ]:
N_SEMANA, N_SEMANAS = CLIENTES_SEMANA, 16

bandit = BanditSegmentado(ASSUNTOS, SEGMENTOS, segmentar, random_state=7)
sim = Simulador(2024)

historico, coorte_verde, log_completo = [], None, []

for semana in range(1, N_SEMANAS + 1):
    ctx = sim.gerar(N_SEMANA)

    # --- Holdout: sorteado na abertura da conta, ANTES do bandit ------------
    # Regras de negócio (elegibilidade, opt-out, frequência) entram aqui também,
    # filtrando quem chega ao motor — nunca como pós-filtro da ação escolhida.
    no_holdout = sim.rng.random(N_SEMANA) < FRACAO_HOLDOUT
    fluxo = ctx.loc[~no_holdout]

    # --- Decisão: a tabela é fixada para o lote inteiro ---------------------
    tabela = bandit.tabela_politica()
    acoes, decisao = bandit.recommend_batch(fluxo, tabela=tabela)

    # --- O que vai para o data warehouse -----------------------------------
    lote = ctx.join(decisao.drop(columns="segmento"))     # propensão de todos os braços
    lote["grupo"] = np.where(no_holdout, HOLDOUT, "fluxo")
    lote["acao"] = HOLDOUT
    lote.loc[fluxo.index, "acao"] = acoes
    lote["semana"] = semana
    lote["model_version"] = f"ts-v{semana:03d}"
    lote["cliente_id"] = f"s{semana:02d}_" + lote.index.astype(str)

    # --- ... 7 dias se passam; o resultado volta ---------------------------
    lote["converteu"] = sim.simular(lote["segmento"], lote["acao"])

    # --- Métricas ANTES do treino: refletem a política que decidiu a semana -
    seg_f = fluxo["segmento"]
    otimo = UPLIFT_PP.loc[seg_f].idxmax(axis=1).to_numpy()
    guloso = bandit.melhor_assunto().reindex(seg_f).to_numpy()
    teto_s = sim.taxa_esperada(seg_f, otimo)
    piso_s = float(BASE.reindex(seg_f).to_numpy().mean())
    ganho = lambda a: (sim.taxa_esperada(seg_f, a) - piso_s) / (teto_s - piso_s)

    historico.append(dict(
        semana=semana,
        base_treino=int((bandit.succ + bandit.fail).sum()),
        realizado=ganho(acoes),
        aprendido=ganho(guloso),
        acerto=(guloso == otimo).mean(),
        conv_fluxo=lote.loc[lote.grupo == "fluxo", "converteu"].mean(),
        conv_hold=lote.loc[lote.grupo == HOLDOUT, "converteu"].mean(),
    ))

    # --- Job semanal: treina na coorte que MADUROU (a de 7+ dias atrás) ----
    log_completo.append(lote)
    if coorte_verde is not None:
        bandit.update(coorte_verde, verbose=False)
    coorte_verde = lote

print("realizado/aprendido = % do ganho máximo possível sobre não conversar sobre nada")
print(pd.DataFrame(historico).to_string(index=False, formatters={
    "realizado": "{:.0%}".format, "aprendido": "{:.0%}".format,
    "acerto": "{:.0%}".format, "base_treino": "{:,}".format,
    "conv_fluxo": "{:.4f}".format, "conv_hold": "{:.4f}".format,
}))

### A política aprendida vs. a verdade

A tabela abaixo **é** o modelo. É isso que vai para produção: nenhum objeto serializado,
nenhuma feature calculada na requisição — 24 números e um lookup por segmento.

`p_melhor` é a leitura que serve para conversar com stakeholder: *"temos 99% de certeza
de que pix é o melhor assunto para jovem de renda baixa"*. E `post_sd` na casa de
0,2 p.p. contra diferenças verdadeiras de 0,6 a 3 p.p. é a explicação de por que o motor
resolveu os segmentos fáceis em 3 semanas e os apertados levaram até a 10ª.

In [ ]:
print("TABELA DE POLÍTICA (probabilidade de servir cada assunto) — o artefato de deploy")
print(bandit.tabela_politica().round(3).to_string())

comparacao = pd.DataFrame({
    "aprendido": bandit.melhor_assunto(),
    "verdadeiro": UPLIFT_PP.idxmax(axis=1),
})
comparacao["ok"] = np.where(comparacao.aprendido == comparacao.verdadeiro, "sim", "NAO")
print("\nMelhor assunto: aprendido vs. verdadeiro")
print(comparacao.to_string())

est = bandit.estado()
print("\nEstado das células com participação relevante (p_melhor > 5%)")
print(est[est.p_melhor > 0.05].round(4).to_string(index=False))

## 6. Validação 3 — o holdout, que é o que substitui o controle interno

Este é o pedaço que o desenho causal resolveria com um braço de controle dentro do
fluxo, e que aqui sai de graça do holdout que vocês já vão ter.

O holdout é aleatório sobre **todos** os que abrem conta, então ele é comparável dentro
de qualquer segmento. Isso dá as duas respostas que interessam ao negócio:

1. **O programa funciona?** Fluxo vs. holdout no total.
2. **Estamos machucando algum segmento?** A mesma conta por segmento. Um segmento com
   lift negativo e IC que não cruza zero é o sinal de "pare de conversar com esse
   perfil" — a decisão de silêncio, obtida sem estimar CATE nenhum.

O que ele **não** dá é "o assunto pix, isolado, gera +X p.p. contra o silêncio". Se
alguém exigir esse número por assunto, a resposta não é o notebook causal inteiro: é um
controle interno de 5-10% e uma diferença de proporções — a última seção detalha.

Repare no veredito `inconcluso` que aparece em algum segmento pequeno na janela curta:
o holdout dele tem poucos milhares de linhas e o IC atravessa o zero. Isso é o
comportamento correto e é a razão de a coluna existir — a leitura é "ainda não sei",
não "não funciona". A seção 7 diz quantas semanas faltam.

In [ ]:
def analise_holdout(log, z=1.96):
    """
    Fluxo vs. holdout: diferença de proporções com IC de 95%.
    Roda no total e por segmento. É a medida oficial de incrementalidade.
    """
    linhas = []
    grupos = [("TOTAL", log)] + list(log.groupby("segmento", observed=True))
    for chave, g in grupos:
        f = g.loc[g["grupo"] == "fluxo", "converteu"]
        h = g.loc[g["grupo"] == HOLDOUT, "converteu"]
        if len(f) == 0 or len(h) == 0:
            continue
        p1, p2, n1, n2 = f.mean(), h.mean(), len(f), len(h)
        se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
        linhas.append(dict(
            segmento=chave, n_fluxo=n1, n_holdout=n2,
            taxa_fluxo=p1, taxa_holdout=p2,
            lift_pp=(p1 - p2) * 100,
            ic95_baixo=(p1 - p2 - z * se) * 100,
            ic95_alto=(p1 - p2 + z * se) * 100,
        ))
    out = pd.DataFrame(linhas)
    out["veredito"] = np.where(out.ic95_baixo > 0, "ganha",
                        np.where(out.ic95_alto < 0, "MACHUCA", "inconcluso"))
    return out


log = pd.concat(log_completo, ignore_index=True)
print(f"Log acumulado: {len(log):,} linhas / {N_SEMANAS} semanas\n")
print("INCREMENTALIDADE — fluxo vs. holdout (acumulado)")
print(analise_holdout(log).round(3).to_string(index=False))

print("\nMesma conta só nas últimas 6 semanas (política já convergida)")
print(analise_holdout(log[log.semana > N_SEMANAS - 6]).round(3).to_string(index=False))

## 7. Dimensionamento: quantos segmentos e qual piso

Os dois únicos parâmetros que realmente importam no MVP. Ambos têm resposta numérica
com os volumes de `contexto.md`, e ambos são a mesma troca: granularidade e capacidade
de medir custam volume por célula.

O primeiro quadro é o pedágio da exploração. O segundo é o teste de realidade da
granularidade — e é o argumento mais forte contra sair direto para um modelo em features
contínuas: se **24 células** já levam 24 semanas para separar dois assuntos com 2 p.p.
de diferença, uma árvore que precisa *descobrir* o corte sozinha, num alvo com IPW,
precisa de bem mais que isso.

(O quadro assume alocação uniforme — pior caso. O TS chega antes porque realoca sozinho:
ele não precisa *provar* a diferença para começar a explorar o braço melhor. Use os
números como piso de expectativa, não como previsão.)

In [ ]:
media_arms = float((UPLIFT_PP.mean(axis=1) * peso).sum())

print("CUSTO DO PISO DE EXPLORAÇÃO (com a política já convergida)")
print(f"{'piso/braço':>11}{'tráfego unif.':>15}{'ganho':>10}{'% do teto':>11}{'ativações/sem':>16}")
for f in [0.0, 0.02, 0.05, 0.10, 0.15]:
    eps = f * len(ASSUNTOS)
    g = (1 - eps) * teto + eps * media_arms
    print(f"{f:>11.0%}{eps:>15.0%}{g:>9.2f}pp{g / teto:>11.0%}"
          f"{por_semana(g):>16,.0f}")
print(f"(teto = {teto:.2f}pp; média dos 4 assuntos = {media_arms:.2f}pp. "
      f"O piso de 5% custa ~{por_semana(teto - (1 - .2) * teto - .2 * media_arms):,.0f} "
      f"ativações/semana — o preço de nunca ficar cego.)")


def semanas_para_provar(gap_pp, n_seg, base=TAXA_D7, n_arms=len(ASSUNTOS),
                        n_semana=CLIENTES_SEMANA, holdout=FRACAO_HOLDOUT,
                        z_a=1.96, z_b=0.84):
    """Semanas para separar dois assuntos numa célula, com 80% de poder."""
    obs_celula_semana = n_semana * (1 - holdout) / n_seg / n_arms
    n_req = 2 * (z_a + z_b) ** 2 * base * (1 - base) / (gap_pp / 100) ** 2
    return n_req / obs_celula_semana


print("\nSEMANAS PARA PROVAR UMA DIFERENÇA ENTRE DOIS ASSUNTOS DENTRO DE UMA CÉLULA")
print(f"{'n_segmentos':>12}{'obs/célula/sem':>16}" +
      "".join(f"{f'gap {g}pp':>11}" for g in [3, 2, 1]))
for ns in [4, 6, 8, 12, 24]:
    obs = CLIENTES_SEMANA * (1 - FRACAO_HOLDOUT) / ns / len(ASSUNTOS)
    print(f"{ns:>12}{obs:>16,.0f}" +
          "".join(f"{semanas_para_provar(g, ns):>11.1f}" for g in [3, 2, 1]))

print("\nE O EFEITO DE PROGRAMA (fluxo 80% vs. holdout 20%), que é bem mais barato")
for g in [0.5, 1.0, 1.5, 2.0]:
    n_req = ((1.96 + 0.84) ** 2 * TAXA_D7 * (1 - TAXA_D7)
             * (1 / (1 - FRACAO_HOLDOUT) + 1 / FRACAO_HOLDOUT) / (g / 100) ** 2)
    print(f"  lift de {g:>4.1f} pp  ->  {n_req / CLIENTES_SEMANA:>5.1f} semanas")

## 8. O caminho de produção

Duas peças, e nenhuma delas precisa do objeto `BanditSegmentado` no ar:

- **Serviço de decisão** (síncrono, na abertura da conta): carrega a tabela de política
  versionada, calcula o segmento, sorteia. Milissegundos, sem dependência de ML.
- **Job semanal** (batch): pega a coorte que completou 7 dias, chama `update()`, publica
  a tabela nova com uma versão nova. Se o job falhar, a tabela antiga continua servindo
  — degradação graciosa de graça.

A célula abaixo mostra o serviço rodando a partir do **JSON da tabela**, sem nenhum
objeto de modelo, para deixar claro o que é o artefato de deploy.

In [ ]:
# ---- O artefato: exporte isso ao fim de cada job de treino ------------------
politica_json = bandit.tabela_politica().to_json(orient="index")
print("politica_ts-v016.json (o deploy inteiro):")
print(politica_json[:220] + " ...\n")


# ---- O serviço de decisão: sem modelo, só a tabela -------------------------
def servir(cliente, politica, assuntos, rng, versao):
    """Decisão síncrona na abertura da conta. Devolve o que gravar no log."""
    seg = segmentar(pd.DataFrame([cliente])).iloc[0]
    if seg not in politica:
        raise ValueError(f"segmento fora da política: {seg}")   # nunca decida no escuro
    p = np.array([politica[seg][a] for a in assuntos])
    p = p / p.sum()
    assunto = rng.choice(assuntos, p=p)
    return {
        "segmento": seg,
        "acao": assunto,
        **{f"p_{a}": float(pi) for a, pi in zip(assuntos, p)},
        "model_version": versao,
        **{f"feat_{k}": v for k, v in cliente.items()},   # snapshot da decisão
    }


politica = json.loads(politica_json)      # é assim que o serviço a recebe
rng_serve = np.random.default_rng(0)

exemplos = [
    ("jovem, renda baixa",  dict(idade=24, renda=2_500.0)),
    ("35 anos, renda alta", dict(idade=35, renda=12_000.0)),
    ("55 anos, renda alta", dict(idade=55, renda=9_000.0)),
    ("55 anos, renda baixa", dict(idade=55, renda=3_000.0)),
]
for nome, cliente in exemplos:
    r = servir(cliente, politica, ASSUNTOS, rng_serve, "ts-v016")
    seg = r["segmento"]
    preferido = max(ASSUNTOS, key=lambda a: politica[seg][a])
    verdade = UPLIFT_PP.loc[seg]
    nota = "" if r["acao"] == preferido else "   <- caiu no piso de exploração"
    print(f"{nome:<21} seg={seg}")
    print(f"{'':<21} política prefere: {preferido} (p={politica[seg][preferido]:.2f})")
    print(f"{'':<21} sorteado:         {r['acao']} (p={r['p_' + r['acao']]:.2f}){nota}")
    print(f"{'':<21} verdade: melhor={verdade.idxmax()} ({verdade.max():+.1f}pp), "
          f"pior={verdade.idxmin()} ({verdade.min():+.1f}pp)")

print("\nQuando o sorteio não bate com o preferido, é o piso de 5% funcionando: ~20% dos\n"
      "clientes recebem um assunto exploratório para a medição não morrer. O assunto\n"
      "sorteado é o que vai para o LLM gerar a mensagem — o agente personaliza o tom,\n"
      "o motor fixa o assunto. A linha do log é gravada na DECISÃO, não no envio.")

## 9. O que falta antes de ir para produção

**Integridade do log** — é aqui que esse tipo de projeto morre, não no algoritmo.
- Gravar a linha na **decisão**, não no envio. Se a mensagem falhar (erro do agente,
  cliente sem canal), a linha ainda existe e precisa de um flag `entregue`. Contar uma
  não-entrega como tratamento contamina a célula.
- Gravar a linha do **holdout**. Não houve envio nem evento — e sem ela não existe
  medida de incrementalidade nenhuma.
- **Features do momento da decisão.** O `segmento` vai gravado no log e o `update()` não
  recalcula nada, mas isso só protege se quem gravou usou o cadastro daquele instante.
- `model_version` em toda linha. Sem isso não se explica uma queda.

**Operação**
- **Regras de negócio antes do sorteio**, filtrando quem chega ao motor. Como pós-filtro
  da ação escolhida, a propensão logada deixa de bater com o que foi executado.
- **Janela de maturação como cláusula do job**: nunca treinar em coorte com menos de 7
  dias completos. Em produção prefira coorte diária (uma matura por dia) a semanal — a
  atualização fica mais suave e o código é o mesmo.
- Alerta em célula com volume anômalo: quase sempre é bug a montante, não sazonalidade.
- `decay < 1` quando a mensagem em si mudar. Trocar o prompt do agente muda o efeito do
  braço; o posterior não sabe disso e vai levar semanas para esquecer a versão antiga.

**Estatística**
- O ganho reportado ao negócio é sempre **fluxo vs. holdout**, nunca a taxa do fluxo
  isolada e nunca a comparação entre braços (que é enviesada pela própria alocação
  adaptativa — o TS manda mais tráfego para o braço que parece bom, então a média bruta
  por braço tem viés de seleção).
- Comparação entre assuntos: apenas **dentro do segmento**. Entre segmentos, as taxas
  base diferem (0,12 a 0,20 no simulador) e a comparação não quer dizer nada.
- Ativação em D7 é a recompensa; D0 (5%, ~28% das ativações) serve como sinal de
  sanidade rápido para detectar bug de pipeline, **não** como recompensa do bandit.

## Quando graduar para a versão causal

O `bandit.ipynb` deste repositório é o destino, não o ponto de partida. Ele se paga
quando um destes acontecer:

| gatilho | o que muda |
|---|---|
| **Silêncio vira ação real no fluxo** (custo de SMS, risco de descadastro, pressão de frequência) | aí sim é preciso estimar assunto-vs-nada por cliente, e o holdout de 20% não resolve porque ele não é acionável cliente a cliente |
| **Alguém exige uplift por assunto** contra o silêncio | não precisa do notebook causal inteiro: controle interno de 5-10% + diferença de proporções. A seção 7 mostra que 5% dá o ATE por braço em ~3 semanas |
| **Segmentos param de dar conta** — dezenas de células, ou features contínuas com sinal claro | X-learner ou DR-learner (`econml`, `causalml`), não o transformed outcome puro, que é não-enviesado mas ruidoso demais para efeito de 1-2 p.p. |
| **Mais rotas de ativação entram** (o item de `contexto.md`) | cada rota nova é um braço; com 8-10 braços as células ficam finas e vale trocar segmento por modelo |

Nos três primeiros casos, a migração é barata **porque este notebook já loga a propensão
completa de todos os braços**. O histórico gerado pelo MVP é dado interventional válido
para treinar a versão causal — que é justamente o que hoje não existe.